# Intro & Setup

## What's covered

- What Python is, and where it earns its niche against Go, Rust, Java, and JavaScript
- The reference implementation (CPython) versus the alternatives (PyPy, MicroPython, etc.)
- The big language milestones — Python 2 is dead, Python 3.10+ is the target
- How a `.py` file runs — source → bytecode → CPython VM
- Installing Python the right way on macOS — `pyenv` over `brew install python`
- **Virtual environments** — the most important hygiene rule in the language
- `venv`, `pip`, and `pip install -r requirements.txt`
- Modern alternatives — `uv` (fast all-in-one), `poetry`, `pip-tools`
- The IPython REPL and Jupyter — for exploration and notebooks
- Three on-ramps to running code: `python -i`, a script, a project
- The Python ecosystem map — what `requests` is to HTTP, `pandas` to data, `pytest` to tests, etc.
- When NOT to reach for Python

## What Python is

Python is a **dynamically typed, interpreted, multi-paradigm** language designed by Guido van Rossum in 1989. It runs anywhere there is a C compiler, which is to say almost everywhere — Linux, macOS, Windows, embedded boards, scientific clusters, browsers (via Pyodide), and inside large products from Instagram to Dropbox to Industrial Light & Magic.

Four traits that define Python in one breath:

- **Dynamically typed.** Types are attached to *values*, not to *variables*. The same name `x` can hold an integer this line and a string the next. The interpreter checks types at runtime, not at compile time.
- **Interpreted (with a bytecode step underneath).** You don't compile to a machine binary. The CPython runtime parses your source into bytecode and walks it; no separate build step.
- **Multi-paradigm.** Procedural by default. Object-oriented when you want classes. Functional-ish with `lambda`, `map`, `filter`, list comprehensions. Pick the shape that fits.
- **Batteries included.** A large standard library — `os`, `pathlib`, `json`, `datetime`, `csv`, `urllib`, `subprocess`, `re`, `collections`, `itertools`, `functools` — covers most everyday needs without a single `pip install`.

## Where Python sits next to its neighbours

A short comparison against the four languages this user is most likely coming from or going to:

| Versus | Python's pitch |
|---|---|
| **Go** | Far better libraries for data, machine learning, scripting, and integration. Trades Go's compiled-binary deployment story and goroutine concurrency for Python's REPL ergonomics and ecosystem depth. |
| **Rust** | Orders of magnitude less ceremony for prototyping and data work. No borrow checker, no compile times. Trades raw performance and memory safety for productivity. Reach for Rust when Python is too slow; reach for Python when you don't know yet whether speed will matter. |
| **Java** | Less ceremony, faster startup, better data-science tooling. Trades the JVM's mature concurrency story and tooling for Python's lighter touch. The two coexist in many companies — Java for services, Python for the analysis and glue around them. |
| **JavaScript** | A real standard library, a real type-hint story (Python 3.10+), no `undefined` vs `null` confusion, no module-system schism. Trades the browser as a deployment target for Python's better fit on the server side. |

Python's centre of gravity is **data work, scripting, and glue code**. If your job involves analysing data, automating an operating-system task, calling several APIs and stitching results, or training a machine-learning model — Python is almost certainly the right call. If your job is to ship a binary CLI, a desktop app, or a high-throughput web service from scratch in 2026, look harder before reaching for it.

## CPython, PyPy, and friends — "Python" is a language, not an implementation

When you `brew install python` you get **CPython** — the reference implementation, written in C, maintained by the Python core team, and used by ninety-nine percent of Python code in production. When people say "Python" without qualifying, they usually mean CPython.

There are alternatives, each filling a niche:

- **PyPy** — a Python interpreter written in (a subset of) Python, with a just-in-time compiler. Runs many pure-Python programs several times faster than CPython. Compatibility with C-extension libraries (NumPy, pandas) is good but not perfect.
- **MicroPython** and **CircuitPython** — Python for microcontrollers. Run on tiny devices with kilobytes of RAM.
- **Pyodide** — CPython compiled to WebAssembly, runs in a browser. Powers in-browser notebooks like JupyterLite.
- **GraalPy** — Python on the GraalVM, interoperates with Java code.
- **Jython** and **IronPython** — older projects targeting the JVM and .NET. Both are essentially dormant; do not start new projects against them.

For this track, assume CPython. Mentioned here so the names don't surprise you.

## Python 2 is dead. Python 3.10+ is the target.

Python 2 reached end-of-life on January 1, 2020. It receives no security updates. Any new code should target Python 3. If you encounter Python 2 code in the wild, treat it as a port-or-rewrite candidate.

Within Python 3, every minor version brings real changes. The releases that materially affect what you can write:

- **3.6** — f-strings (`f"hello {name}"`). The single most common feature you'll use.
- **3.7** — built-in dict preserves insertion order (was a CPython detail; now a language guarantee). `dataclasses`.
- **3.8** — walrus operator (`:=`).
- **3.9** — built-in generic types (`list[int]` instead of `List[int]`).
- **3.10** — structural pattern matching (`match`/`case`), parenthesized context managers, better error messages.
- **3.11** — significant interpreter speedup (10–60% faster), exception groups, `tomllib`.
- **3.12** — type parameter syntax for generics, `f-string` flexibility.
- **3.13** — experimental no-GIL build, JIT.

This track assumes **Python 3.10 or later**. The `match`/`case` shapes in notebook 03 are 3.10+, and modern type hints in notebook 08 assume 3.10+ syntax.

## How a `.py` file actually runs

Python is "interpreted" but there is a compile step you can't avoid. The full pipeline:

```text
   hello.py
       |
       v
   [ parser ]              <- source -> AST (abstract syntax tree)
       |
       v
   [ compiler ]            <- AST -> bytecode (.pyc cached on disk)
       |
       v
   [ CPython VM ]          <- bytecode interpreter, written in C
       |
       v
     output
```

Three consequences worth absorbing now:

- **Startup includes parse + compile.** The first run of a script reparses; subsequent runs hit the `__pycache__/` directory for cached bytecode and skip the parse. This is why importing a large library is slow only the first time per Python version.
- **The bytecode is real and inspectable.** `import dis; dis.dis(my_function)` shows you the bytecode. Most Python programmers will go their whole careers without needing this — but knowing the layer exists demystifies a lot.
- **The CPython VM is a single-threaded loop.** This is the well-known "GIL" — the Global Interpreter Lock. Only one thread executes Python bytecode at a time. Notebook 09 covers the implications for concurrency.

## Installing Python — `pyenv`, not `brew install python`

The macOS default is to install Python with Homebrew (`brew install python`). This works for casual use but creates real problems the moment you have more than one project:

- Different projects need different Python versions (3.10 here, 3.12 there).
- Upgrading the system Python can break older projects that pinned a version.
- A new macOS release ships its own Python that conflicts.

The right tool is **`pyenv`**, a Python version manager. It installs each version side by side, lets you pin one per directory, and never touches the system Python.

```bash
# install pyenv
brew install pyenv

# install a specific Python version
pyenv install 3.12.4

# set it as the global default
pyenv global 3.12.4

# OR set it per-directory (creates a .python-version file)
cd ~/Projects/myproject
pyenv local 3.12.4
```

Run `python --version` and you should see the version you picked. Run `which python` and the path should be inside `~/.pyenv/`, not `/usr/bin/` or `/opt/homebrew/bin/`.

**If you have already used `brew install python`:** it does no harm, but reach for `pyenv` going forward and let the brew version stay where it is.

## Virtual environments — the most important hygiene rule

A **virtual environment** (or *venv*) is an isolated Python installation, scoped to one project. It contains:

- Its own `python` binary (usually a symlink back to the version pyenv installed).
- Its own `pip`.
- Its own `site-packages/` directory where installed libraries live.

Why this matters: without a venv, every `pip install` modifies your *global* Python install. Project A wants `requests==2.28`, Project B wants `requests==2.32`, you install one, the other breaks. The mess compounds. With a venv per project, each one has its own dependency closure and they cannot interfere.

**The rule: every Python project gets its own virtual environment, always, no exceptions.**

The standard library ships `venv` (since Python 3.3) — no install needed. Create and activate one with two commands.

In [ ]:
%%bash
# Create a venv in the .venv/ folder (the convention)
python -m venv .venv

# Activate it (shell-specific)
source .venv/bin/activate         # bash, zsh
# source .venv/bin/activate.fish  # fish shell
# .venv\Scripts\activate         # Windows PowerShell

# Confirm — python and pip now point inside .venv
which python
which pip

# Deactivate when you're done
deactivate

**What activation actually does.** It prepends `.venv/bin/` to your `PATH`, so when you type `python` you get the one in the venv instead of the system one. It also sets `VIRTUAL_ENV` so tools like `pip` know which environment they're in. Nothing magic — and `deactivate` simply undoes it.

**Convention.** Name the directory `.venv` (with the leading dot) at the project root. Add `.venv/` to `.gitignore` — virtual environments are local and machine-specific, never committed.

**You'll forget to activate.** Everyone does. The symptom: `pip install` putting things in the wrong place, or `import` mysteriously failing. Always check `which python` if something is acting strange — that tells you which environment is in play.

## `pip` and `requirements.txt`

`pip` is Python's package installer. With your venv active, `pip install <package>` downloads from PyPI (the Python Package Index) and installs into the venv's `site-packages/`.

The dependency-declaration convention for simple projects is **`requirements.txt`** — one package per line, with optional version pins. The standard workflow:

```bash
pip install requests          # install a package
pip freeze > requirements.txt # record exact versions

# On another machine, or after blowing away the venv:
pip install -r requirements.txt
```

`pip freeze` writes every installed package and its exact version — including transitive dependencies. That's good for *reproducing* an environment exactly. For declaring what *your* project actually needs, modern projects prefer to list direct dependencies in `pyproject.toml` (see next section) and let a tool like `pip-tools` or `uv` resolve the rest.

In [ ]:
%%bash
# A typical project setup, from scratch, top to bottom

# 1. New project directory
mkdir myproject && cd myproject

# 2. Tell pyenv which Python version
pyenv local 3.12.4

# 3. Create and activate a venv
python -m venv .venv
source .venv/bin/activate

# 4. Install the libraries you need
pip install requests pytest pandas

# 5. Record them for repeatability
pip freeze > requirements.txt

# 6. Verify the world is sane
python --version
which python
pip list

## Modern alternatives — `uv`, `poetry`, `pip-tools`

`venv` + `pip` + `requirements.txt` is the lowest common denominator. As of 2026 there are better tools — slower to learn but worth knowing.

- **`uv`** (Astral) — written in Rust, ten to a hundred times faster than `pip`. Replaces `venv`, `pip`, `pip-tools`, and `pyenv` in one binary. `uv venv && uv pip install ...` is the same workflow but faster. Strong recommendation for new projects.
- **`poetry`** — full project manager. Uses `pyproject.toml` for declared dependencies and `poetry.lock` for resolved ones. Opinionated. Popular in libraries and modern services.
- **`pip-tools`** — separates "what you want" (`requirements.in`) from "what was resolved" (`requirements.txt`). Lower friction than `poetry`, more discipline than raw `pip`.
- **`conda` / `mamba`** — the data-science / machine-learning standard. Manages both Python *and* non-Python dependencies (CUDA, BLAS, native libraries). Reach for it when working with the SciPy stack or ML frameworks.

For this track, `venv` + `pip` is enough. When you start a real project in 2026, default to **`uv`** unless your team has already picked something else.

## The Python REPL, IPython, and Jupyter

Three ways to run Python interactively, in order of feature richness:

- **`python`** (no arguments) — the built-in REPL. Spartan. No syntax highlighting, no tab completion of object attributes, no multi-line editing past the basics. Useful for a one-line check; not for exploration.
- **`ipython`** — installed via `pip install ipython`. Same Python, much better front-end. Tab completion, syntax highlighting, `?` for inline help, `??` for the source code, magic commands (`%time`, `%timeit`, `%run`), reach into the shell with `!ls`. The right default for command-line exploration.
- **Jupyter notebooks** — `pip install jupyter` then `jupyter lab`. Web-based. Cells alternate between markdown and code. Outputs (text, tables, plots) render inline. The right tool for this track and for most exploratory data work.

These notebooks you are reading are Jupyter notebooks. The cells with code in them run in a Jupyter kernel — a Python interpreter listening for cell submissions.

## Three on-ramps to running code

Pick by what you are doing.

```text
   +-------------+   +---------------+   +------------------+
   |    REPL     |   |    script     |   |     project      |
   |  (one-off)  |   |   one file    |   |   many files,    |
   +-------------+   +---------------+   +------------------+
    ipython         python script.py    package + venv
    quick check     run from cli        installable, tested
```

- **REPL** for "what does this expression return?" — IPython.
- **Script** for a one-file utility. `python myscript.py`. Optionally make it executable with a `#!/usr/bin/env python3` shebang on the first line.
- **Project** for anything with multiple files, dependencies, tests, or that you'll publish or deploy. Has a `pyproject.toml`, a `.venv`, a `tests/` directory, often a `src/` layout.

## `python -i` — the hybrid that's underused

`python -i script.py` runs `script.py` to completion and then **drops you into a REPL with everything the script defined still in scope**. The fastest possible "let me poke at this" loop:

```bash
python -i my_data_loader.py
# script runs, then >>> prompt appears with df, my_function, etc. all available
>>> df.head()
>>> my_function(42)
```

This is the closest CPython gets to the Common Lisp / SLIME experience. Combined with IPython (`ipython -i script.py`), it's an underrated workflow.

## A minimal project layout

When you graduate past a single script, the conventional Python project looks like this:

```text
   myproject/
   ├── pyproject.toml      <- project metadata, dependencies, build config
   ├── README.md
   ├── .gitignore           <- include `.venv/`, `__pycache__/`, `*.pyc`
   ├── .python-version      <- pyenv pin
   ├── src/
   │   └── myproject/       <- the actual importable package
   │       ├── __init__.py
   │       └── core.py
   └── tests/
       └── test_core.py
```

The **src layout** (package inside `src/`) prevents a class of import bugs where you accidentally import the local source instead of the installed package. Notebook 06 covers the `__init__.py`, the `from myproject.core import x` mechanics, and packaging.

For now: know that this is the target shape. Single-file scripts are fine for quick work; reach for this layout when the project starts to feel non-trivial.

## The ecosystem map — what library does what

A short map so you can recognize names when they come up.

| Area | Standard library | Most-reached-for third-party |
|---|---|---|
| HTTP client | `urllib.request` | `requests` (sync), `httpx` (sync + async) |
| HTTP server | `http.server` | `fastapi`, `flask`, `starlette` |
| Data — tabular | `csv`, `sqlite3` | `pandas`, `polars` |
| Data — numeric | `array` | `numpy`, `scipy` |
| Machine learning | — | `scikit-learn`, `pytorch`, `tensorflow` |
| Plotting | — | `matplotlib`, `plotly`, `seaborn` |
| JSON | `json` | (stdlib is enough) |
| YAML | — | `pyyaml`, `ruamel.yaml` |
| TOML (read) | `tomllib` (3.11+) | — |
| TOML (write) | — | `tomli-w` |
| Testing | `unittest` | `pytest` (notebook 10) |
| Type checking | — | `mypy`, `pyright`, `ruff` |
| Linting / formatting | — | `ruff`, `black` |
| Async runtime | `asyncio` | `trio`, `anyio` |
| CLI | `argparse` | `click`, `typer` |
| Date / time | `datetime`, `zoneinfo` | `pendulum`, `arrow` |
| Templating | `string.Template` | `jinja2` |

You will install only the ones your project needs. The list is here so the names are not a surprise.

## When NOT to use Python

Four honest cases where reaching for Python is the wrong call.

- **Hard real-time or sub-millisecond latency.** Python's garbage collector and interpreter loop introduce unpredictable pauses. Reach for Go, Rust, or C++ when latency matters.
- **CPU-bound work that needs to scale across cores.** The GIL serializes pure-Python execution. You can work around it with `multiprocessing`, with C extensions, or with native libraries that release the GIL (numpy does, pandas mostly does), but if the work is heavy and pure-Python, Python is the wrong shape.
- **Distribution as a single binary.** PyInstaller and similar tools can bundle a Python program, but the result is a forty-megabyte file that has to extract itself before running. Go and Rust produce true single-file binaries.
- **Mobile.** Python on Android and iOS is possible (Beeware, Kivy) but a small niche. The dominant choices are Kotlin, Swift, React Native, Flutter.

The honest rule: Python earns its place when you'd rather optimise for *time-to-result* than *time-to-execute*. The moment execution time becomes the bottleneck, look at what the slow path actually is — usually you can fix it with NumPy, with Cython, with a C extension, or by replacing one hot loop with a call into Rust. Rarely do you need to rewrite the whole program in another language.

## What's next

You now have Python installed, a venv created, the dependency tooling demystified, and the ecosystem map.

Notebook 02 starts the language proper — the values that Python lets you create, the types those values have, and the operators that combine them. The two new mental shifts coming from a statically typed background are that variables hold *references* to objects, not the objects themselves, and that almost everything in Python — including types, functions, and modules — is itself an object you can pass around.